# CosmoMamba: State-Space Models for Cosmological Parameter Inference

**First application of selective state-space models (Mamba) to cosmological field analysis.**

Benchmarks CosmoMamba against CNNs and Vision Transformers on the CAMELS Multifield Dataset (CMD).

---

**How to run on Kaggle (use committed runs, NOT interactive):**

1. Settings: Accelerator > **GPU T4 x2**, Internet > **On**
2. Click **Save & Run All (Commit)** — runs in background for up to 12 hours
3. Close your browser. Check progress anytime from the Versions tab.
4. If it doesn't finish: go to the output of that version, click "New Dataset",
   name it `cosmo-mamba-checkpoints`, then add it as input to this notebook and commit again.
   Training auto-resumes from where it stopped.

Results save to `/kaggle/working/` which becomes the version output.

## 0. Setup

In [ ]:
!pip install -q einops

import os, sys, time, json, copy, math, warnings, shutil, subprocess
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from einops import rearrange
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-detect platform
if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
elif Path('/content').exists():
    WORK_DIR = Path('/content')
else:
    WORK_DIR = Path('.')

DATA_DIR = WORK_DIR / 'cmd_data'
RUNS_DIR = WORK_DIR / 'runs'
DATA_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

# Restore data + checkpoints from any input dataset
INPUT = Path('/kaggle/input')
if INPUT.exists():
    print('Scanning /kaggle/input/ for data and checkpoints...')
    # Show what Kaggle mounted so we can debug if needed
    for d in sorted(INPUT.glob('*')):
        items = list(d.rglob('*'))
        files = [x for x in items if x.is_file()]
        print(f'  {d.name}/ ({len(files)} files)')

    # Restore data: any .npy or .txt file with Maps_ or params_ in the name
    for f in INPUT.rglob('*'):
        if not f.is_file() or f.stat().st_size < 100:
            continue
        if f.suffix in ('.npy', '.txt') and ('Maps_' in f.name or 'params_' in f.name):
            dest = DATA_DIR / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                print(f'  Restored data: {f.name} ({f.stat().st_size/1e6:.1f} MB)')

    # Restore checkpoints: any .pt file that's inside a model-named folder
    for f in INPUT.rglob('*.pt'):
        if f.parent.name in ('runs',) or not f.parent.name:
            continue
        dest = RUNS_DIR / f.parent.name / f.name
        dest.parent.mkdir(exist_ok=True)
        if not dest.exists():
            shutil.copy2(f, dest)
            print(f'  Restored checkpoint: {f.parent.name}/{f.name}')

print(f'Data dir: {list(DATA_DIR.glob("*"))}')
print(f'Checkpoints: {list(RUNS_DIR.rglob("*.pt"))}')
print()

print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

## 1. Download CMD Data

Downloads dark matter density (Mcdm) maps from IllustrisTNG and SIMBA.

**If downloads fail** (0 byte files), use METHOD B:
1. Download the 4 files on your local machine (curl commands printed below)
2. Upload as a Kaggle Dataset named `cmd-mcdm`
3. Add it via sidebar > Add Data
4. Uncomment the `DATA_DIR` override at the bottom of the download cell

In [ ]:
BASE_URL = (
    'https://users.flatironinstitute.org/~fvillaescusa/priv/'
    'DEPnzxoWlaTQ6CjrXqsm0vYi8L7Jy/CMD/2D_maps/data'
)

# Clean up 0-byte files from any previous failed attempt
for f in DATA_DIR.glob('*'):
    if f.stat().st_size == 0:
        print(f'Removing empty file: {f.name}')
        f.unlink()

def download_file(fname, suite, min_bytes=100):
    dest = DATA_DIR / fname
    if dest.exists() and dest.stat().st_size > min_bytes:
        print(f'  Cached: {fname} ({dest.stat().st_size/1e6:.1f} MB)')
        return True
    if dest.exists():
        dest.unlink()
    suite_subdir = 'Nbody' if suite.startswith('Nbody') else suite
    url = f'{BASE_URL}/{suite_subdir}/{fname}'
    print(f'  Downloading {fname} ...', flush=True)
    try:
        # curl with -L (follow redirects), -f (fail on HTTP errors)
        subprocess.run(
            ['curl', '-L', '-f', '--progress-bar', '-o', str(dest), url],
            check=True, timeout=1200,
        )
        size = dest.stat().st_size
        if size > min_bytes:
            print(f'  OK: {size/1e6:.1f} MB')
            return True
        print(f'  ERROR: only {size} bytes')
        dest.unlink()
        return False
    except Exception as e:
        print(f'  curl failed ({e}), trying urllib...')
        if dest.exists():
            dest.unlink()
    # Fallback: Python urllib
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=600) as resp:
            with open(dest, 'wb') as f:
                shutil.copyfileobj(resp, f, length=1024*1024)
        size = dest.stat().st_size
        if size > min_bytes:
            print(f'  OK (urllib): {size/1e6:.1f} MB')
            return True
        print(f'  FAILED: only {size} bytes')
        if dest.exists(): dest.unlink()
        return False
    except Exception as e:
        print(f'  FAILED: {e}')
        if dest.exists(): dest.unlink()
        return False

all_ok = True
for suite in ['IllustrisTNG', 'SIMBA']:
    print(f'\n--- {suite} ---')
    all_ok &= download_file(f'params_LH_{suite}.txt', suite)
    all_ok &= download_file(f'Maps_Mcdm_{suite}_LH_z=0.00.npy', suite)

print()
for f in sorted(DATA_DIR.glob('*')):
    print(f'  {f.name:50s} {f.stat().st_size/1e6:>8.1f} MB')

if all_ok:
    print('\nAll downloads OK!')
else:
    print(f'''
{'='*62}
  Downloads failed. Use METHOD B:

  1. On your LOCAL machine run:
     curl -LO "{BASE_URL}/IllustrisTNG/params_LH_IllustrisTNG.txt"
     curl -LO "{BASE_URL}/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy"
     curl -LO "{BASE_URL}/SIMBA/params_LH_SIMBA.txt"
     curl -LO "{BASE_URL}/SIMBA/Maps_Mcdm_SIMBA_LH_z=0.00.npy"

  2. kaggle.com/datasets > New Dataset > upload those 4 files
     Name it "cmd-mcdm"

  3. In this notebook: sidebar > Add Data > search "cmd-mcdm"

  4. Uncomment the line below and re-run.
{'='*62}
''')

# UNCOMMENT this line if using METHOD B (Kaggle Dataset upload):
# DATA_DIR = Path('/kaggle/input/cmd-mcdm')

## 2. Dataset & Data Loading

In [ ]:
MAPS_PER_SIM = 15

class CMDDataset(Dataset):
    def __init__(self, data_dir, fields, suite='IllustrisTNG', split='train',
                 train_frac=0.8, val_frac=0.1, normalize=True, augment=False, seed=42):
        self.fields = fields
        self.normalize = normalize
        self.augment = augment and (split == 'train')
        params_file = os.path.join(data_dir, f'params_LH_{suite}.txt')
        self.params = np.loadtxt(params_file)
        n_sims = len(self.params)
        n_maps = n_sims * MAPS_PER_SIM
        self.maps = {}
        self.field_stats = {}
        for field in fields:
            fpath = os.path.join(data_dir, f'Maps_{field}_{suite}_LH_z=0.00.npy')
            maps = np.load(fpath, mmap_mode='r')
            self.maps[field] = maps
            if normalize:
                sample = maps[:1000].astype(np.float32)
                nonzero = sample[sample != 0]
                if len(nonzero) > 0:
                    log_data = np.log10(np.abs(nonzero) + 1e-10)
                    self.field_stats[field] = (float(np.mean(log_data)), float(np.std(log_data)))
                else:
                    self.field_stats[field] = (0.0, 1.0)
        rng = np.random.RandomState(seed)
        sim_indices = np.arange(n_sims)
        rng.shuffle(sim_indices)
        n_train = int(n_sims * train_frac)
        n_val = int(n_sims * val_frac)
        if split == 'train':   sims = sim_indices[:n_train]
        elif split == 'val':   sims = sim_indices[n_train:n_train + n_val]
        elif split == 'test':  sims = sim_indices[n_train + n_val:]
        else: raise ValueError(split)
        self.indices = []
        for s in sims:
            for m in range(MAPS_PER_SIM):
                self.indices.append(s * MAPS_PER_SIM + m)

    def __len__(self):
        return len(self.indices)

    def _normalize_map(self, x, field):
        x = x.astype(np.float32)
        x = np.log10(np.abs(x) + 1e-10)
        mean, std = self.field_stats[field]
        if std > 0: x = (x - mean) / std
        return x

    def __getitem__(self, idx):
        map_idx = self.indices[idx]
        sim_idx = map_idx // MAPS_PER_SIM
        channels = []
        for field in self.fields:
            m = self.maps[field][map_idx]
            m = self._normalize_map(m, field) if self.normalize else m.astype(np.float32)
            channels.append(m)
        x = torch.tensor(np.stack(channels, axis=0))
        if self.augment:
            k = torch.randint(0, 4, (1,)).item()
            x = torch.rot90(x, k, dims=(-2, -1))
            if torch.rand(1).item() > 0.5: x = torch.flip(x, dims=(-1,))
            if torch.rand(1).item() > 0.5: x = torch.flip(x, dims=(-2,))
        y = torch.tensor(self.params[sim_idx, :2].astype(np.float32))
        return x, y

def make_loaders(data_dir, fields, suite, batch_size=32, num_workers=2):
    common = dict(data_dir=data_dir, fields=fields, suite=suite)
    train_ds = CMDDataset(**common, split='train', augment=True)
    val_ds   = CMDDataset(**common, split='val',   augment=False)
    test_ds  = CMDDataset(**common, split='test',  augment=False)
    kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True)
    return (
        DataLoader(train_ds, shuffle=True,  drop_last=True, **kw),
        DataLoader(val_ds,   shuffle=False, **kw),
        DataLoader(test_ds,  shuffle=False, **kw),
    )

train_loader, val_loader, test_loader = make_loaders(str(DATA_DIR), ['Mcdm'], 'IllustrisTNG')
x, y = next(iter(train_loader))
print(f'Batch: x={list(x.shape)}, y={list(y.shape)}')
print(f'Splits: train={len(train_loader.dataset)}, val={len(val_loader.dataset)}, test={len(test_loader.dataset)}')
print(f'Omega_m range: [{y[:,0].min():.3f}, {y[:,0].max():.3f}]')
print(f'sigma_8 range: [{y[:,1].min():.3f}, {y[:,1].max():.3f}]')

## 3. Model Definitions

In [ ]:
# 3a. CNN Baseline
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class CNNBaseline(nn.Module):
    def __init__(self, in_channels=1, n_params=2, channels=None, fc=256, dropout=0.5):
        super().__init__()
        channels = channels or [16, 32, 64, 128, 256, 256]
        layers = []
        ch = in_channels
        for i, c in enumerate(channels):
            layers.append(ConvBlock(ch, c))
            if i < len(channels) - 1: layers.append(nn.AvgPool2d(2))
            ch = c
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc), nn.LeakyReLU(0.2), nn.Dropout(dropout),
            nn.Linear(fc, fc), nn.LeakyReLU(0.2), nn.Dropout(dropout))
        self.mean_out = nn.Linear(fc, n_params)
        self.logvar_out = nn.Linear(fc, n_params)
    def forward(self, x):
        h = self.pool(self.features(x)).flatten(1)
        h = self.head(h)
        return self.mean_out(h), self.logvar_out(h)

# 3b. Vision Transformer
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=1, dim=384):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, dim, patch_size, patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class ViTRegressor(nn.Module):
    def __init__(self, in_channels=1, n_params=2, img_size=256, patch_size=16,
                 embed_dim=384, depth=6, num_heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim // 2), nn.GELU(), nn.Dropout(dropout))
        self.mean_out = nn.Linear(embed_dim // 2, n_params)
        self.logvar_out = nn.Linear(embed_dim // 2, n_params)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        tokens = self.patch_embed(x)
        tokens = torch.cat([self.cls_token.expand(B, -1, -1), tokens], dim=1)
        tokens = self.pos_drop(tokens + self.pos_embed)
        cls_out = self.norm(self.encoder(tokens)[:, 0])
        h = self.head(cls_out)
        return self.mean_out(h), self.logvar_out(h)

In [ ]:
# 3c. CosmoMamba (THE NOVEL CONTRIBUTION)

def _parallel_scan(gates, values):
    """Hillis-Steele parallel prefix scan for linear recurrence:
       h[t] = gates[t] * h[t-1] + values[t],  h[-1] = 0.
    Runs in O(log L) parallel steps instead of O(L) sequential steps.
    gates, values: (B, L, ...) with gates in (0, 1)."""
    B, L = gates.shape[:2]
    trailing = gates.shape[2:]
    log2L = int(math.ceil(math.log2(max(L, 2))))
    Lp = 2 ** log2L
    if Lp > L:
        pad_shape = (B, Lp - L) + trailing
        gates = torch.cat([gates, gates.new_ones(pad_shape)], dim=1)
        values = torch.cat([values, values.new_zeros(pad_shape)], dim=1)
    for d in range(log2L):
        k = 2 ** d
        g_shift = torch.cat([gates.new_ones(B, k, *trailing), gates[:, :-k]], dim=1)
        v_shift = torch.cat([values.new_zeros(B, k, *trailing), values[:, :-k]], dim=1)
        values = gates * v_shift + values
        gates = gates * g_shift
    return values[:, :L]

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_state = d_state
        d_inner = d_model * expand
        self.d_inner = d_inner
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv-1, groups=d_inner)
        self.x_proj = nn.Linear(d_inner, d_state * 2 + 1, bias=False)
        self.A_log = nn.Parameter(
            torch.log(torch.arange(1, d_state+1, dtype=torch.float32))
            .unsqueeze(0).expand(d_inner, -1).clone())
        self.D = nn.Parameter(torch.ones(d_inner))
        self.dt_proj = nn.Linear(1, d_inner, bias=True)
        nn.init.constant_(self.dt_proj.bias, -4.0)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_inner)

    def forward(self, x):
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)
        x_branch = rearrange(x_branch, 'b l d -> b d l')
        x_branch = self.conv1d(x_branch)[:, :, :L]
        x_branch = rearrange(x_branch, 'b d l -> b l d')
        x_branch = F.silu(x_branch)
        proj = self.x_proj(x_branch)
        dt_input = proj[..., :1]
        B_param = proj[..., 1:1+self.d_state]
        C_param = proj[..., 1+self.d_state:]
        dt = F.softplus(self.dt_proj(dt_input))
        A = -torch.exp(self.A_log.float())
        dA = torch.exp(dt.float().unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        dB = dt.unsqueeze(-1) * B_param.unsqueeze(-2)
        x_scaled = x_branch.unsqueeze(-1) * dB
        h_all = _parallel_scan(dA, x_scaled.float()).to(x_branch.dtype)
        y = (h_all * C_param.unsqueeze(-2)).sum(-1)
        y = y + x_branch * self.D.unsqueeze(0).unsqueeze(0)
        y = self.norm(y) * F.silu(z)
        return self.out_proj(y)

class MultiDirectionalScan(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.ssm_fwd     = SelectiveSSM(d_model, d_state, d_conv, expand)
        self.ssm_bwd     = SelectiveSSM(d_model, d_state, d_conv, expand)
        self.ssm_col_fwd = SelectiveSSM(d_model, d_state, d_conv, expand)
        self.ssm_col_bwd = SelectiveSSM(d_model, d_state, d_conv, expand)
        self.merge = nn.Linear(d_model * 4, d_model)
    def forward(self, x, H, W):
        B, L, D = x.shape
        y_fwd = self.ssm_fwd(x)
        y_bwd = self.ssm_bwd(x.flip(1)).flip(1)
        x_col = rearrange(x.view(B, H, W, D), 'b h w d -> b (w h) d')
        y_col_fwd = self.ssm_col_fwd(x_col)
        y_col_fwd = rearrange(y_col_fwd.view(B, W, H, D), 'b w h d -> b (h w) d')
        y_col_bwd = self.ssm_col_bwd(x_col.flip(1)).flip(1)
        y_col_bwd = rearrange(y_col_bwd.view(B, W, H, D), 'b w h d -> b (h w) d')
        return self.merge(torch.cat([y_fwd, y_bwd, y_col_fwd, y_col_bwd], dim=-1))

class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, ffn_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.scan = MultiDirectionalScan(d_model, d_state, d_conv, expand)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        ffn = int(d_model * ffn_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn, d_model), nn.Dropout(dropout))
    def forward(self, x, H, W):
        x = x + self.drop1(self.scan(self.norm1(x), H, W))
        x = x + self.ffn(self.norm2(x))
        return x

class CosmoMamba(nn.Module):
    def __init__(self, in_channels=1, n_params=2, img_size=256, patch_size=16,
                 embed_dim=192, depth=8, d_state=16, d_conv=4, expand=2,
                 ffn_ratio=4.0, dropout=0.1):
        super().__init__()
        self.H = img_size // patch_size
        self.W = img_size // patch_size
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim//2, 7, stride=4, padding=3, bias=False),
            nn.BatchNorm2d(embed_dim//2), nn.GELU(),
            nn.Conv2d(embed_dim//2, embed_dim, 3, stride=patch_size//4, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim), nn.GELU())
        n_patches = self.H * self.W
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            MambaBlock(embed_dim, d_state, d_conv, expand, ffn_ratio, dropout)
            for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(dropout))
        self.mean_out = nn.Linear(embed_dim, n_params)
        self.logvar_out = nn.Linear(embed_dim, n_params)

    def forward(self, x):
        x = self.patch_embed(x)
        _, _, H, W = x.shape
        x = rearrange(x, 'b c h w -> b (h w) c')
        if x.shape[1] != self.H * self.W:
            pos = rearrange(self.pos_embed, '1 (h w) c -> 1 c h w', h=self.H, w=self.W)
            pos = F.interpolate(pos, size=(H, W), mode='bilinear', align_corners=False)
            pos = rearrange(pos, '1 c h w -> 1 (h w) c')
        else:
            pos = self.pos_embed
            H, W = self.H, self.W
        x = self.pos_drop(x + pos)
        for block in self.blocks:
            x = block(x, H, W)
        x = self.norm(x).mean(dim=1)
        h = self.head(x)
        return self.mean_out(h), self.logvar_out(h)

# Verify
x_test = torch.randn(2, 1, 256, 256)
for name, M in [('CNN', CNNBaseline), ('ViT', ViTRegressor),
                 ('CosmoMamba', lambda: CosmoMamba(embed_dim=128, depth=6, patch_size=32))]:
    m = M(); out = m(x_test)[0]
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:12s} | {n/1e6:6.2f}M params | output: {list(out.shape)}')
del x_test

## 4. Training Engine

In [ ]:
def gaussian_nll(mean, logvar, target):
    var = torch.exp(logvar).clamp(min=1e-6)
    return 0.5 * (logvar + (target - mean)**2 / var).mean()

def compute_metrics(preds, targets):
    results = {}
    for i, name in enumerate(['Omega_m', 'sigma_8']):
        p, t = preds[:, i], targets[:, i]
        r = p - t
        results[f'{name}_rmse'] = np.sqrt(np.mean(r**2))
        results[f'{name}_r2'] = 1.0 - np.sum(r**2) / (np.sum((t - t.mean())**2) + 1e-8)
        results[f'{name}_rel_err'] = np.mean(np.abs(r) / (np.abs(t) + 1e-8))
    results['mean_r2'] = (results['Omega_m_r2'] + results['sigma_8_r2']) / 2
    return results

def train_model(model, train_loader, val_loader, name,
                epochs=300, lr=1e-4, wd=1e-4, warmup=10, patience=40, grad_clip=1.0):
    run_dir = RUNS_DIR / name
    run_dir.mkdir(exist_ok=True)
    best_ckpt = run_dir / 'best_model.pt'
    resume_ckpt = run_dir / 'resume.pt'

    if best_ckpt.exists() and not resume_ckpt.exists():
        print(f'  {name}: already done — loading checkpoint')
        ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        model.to(DEVICE)
        model.load_state_dict(ckpt['model'])
        return model, ckpt['history']

    model = model.to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=wd)
    warmup_sched = LinearLR(optimizer, start_factor=0.01, total_iters=warmup)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, [warmup_sched, cosine_sched], milestones=[warmup])
    use_amp = (DEVICE.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    start_epoch = 1
    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_r2_om': [], 'val_r2_s8': []}

    if resume_ckpt.exists():
        r = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(r['model_state'])
        optimizer.load_state_dict(r['optim_state'])
        scheduler.load_state_dict(r['sched_state'])
        start_epoch = r['epoch'] + 1
        best_val_loss = r['best_val_loss']
        best_state = r['best_state']
        no_improve = r['no_improve']
        history = r['history']
        print(f'  RESUMED {name} from epoch {r["epoch"]} (best loss: {best_val_loss:.5f})')

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*60}\n{name} ({n_params/1e6:.2f}M) | epochs {start_epoch}-{epochs}\n{"="*60}')
    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=use_amp):
                mean, logvar = model(x)
                loss = gaussian_nll(mean, logvar, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0.0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    mean, logvar = model(x)
                    val_loss += gaussian_nll(mean, logvar, y).item()
                all_preds.append(mean.cpu().numpy())
                all_targets.append(y.cpu().numpy())
        val_loss /= len(val_loader)
        metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_targets))
        scheduler.step()
        elapsed = time.time() - t0
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_r2_om'].append(metrics['Omega_m_r2'])
        history['val_r2_s8'].append(metrics['sigma_8_r2'])
        if epoch <= 5 or epoch % 10 == 0:
            print(f'  Ep {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | '
                  f'R2(Om) {metrics["Omega_m_r2"]:.4f} | R2(s8) {metrics["sigma_8_r2"]:.4f} | {elapsed:.1f}s')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
        if epoch % 10 == 0:
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'sched_state': scheduler.state_dict(),
                'best_val_loss': best_val_loss, 'best_state': best_state,
                'no_improve': no_improve, 'history': history,
            }, resume_ckpt)
        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break
    model.load_state_dict(best_state)
    torch.save({'model': best_state, 'history': history}, best_ckpt)
    if resume_ckpt.exists(): resume_ckpt.unlink()
    print(f'  Done! Best val loss: {best_val_loss:.5f}')
    return model, history

## 5. Evaluation Utilities

In [ ]:
@torch.no_grad()
def test_model(model, loader, label=''):
    model.eval()
    all_preds, all_logvars, all_targets = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        mean, logvar = model(x)
        all_preds.append(mean.cpu().numpy())
        all_logvars.append(logvar.cpu().numpy())
        all_targets.append(y.numpy())
    preds = np.concatenate(all_preds)
    logvars = np.concatenate(all_logvars)
    targets = np.concatenate(all_targets)
    metrics = compute_metrics(preds, targets)
    if label:
        print(f'  [{label}] R2(Om)={metrics["Omega_m_r2"]:.4f} R2(s8)={metrics["sigma_8_r2"]:.4f} '
              f'RMSE(Om)={metrics["Omega_m_rmse"]:.5f} RMSE(s8)={metrics["sigma_8_rmse"]:.5f}')
    return preds, logvars, targets, metrics

def plot_comparison(results_dict, save_path):
    fig, axes = plt.subplots(len(results_dict), 2, figsize=(10, 4*len(results_dict)))
    if len(results_dict) == 1: axes = axes.reshape(1, -1)
    for row, (mname, (preds, logvars, targets, met)) in enumerate(results_dict.items()):
        for col, param in enumerate(['Omega_m', 'sigma_8']):
            ax = axes[row, col]
            p, t = preds[:, col], targets[:, col]
            u = np.sqrt(np.exp(logvars[:, col]))
            ax.errorbar(t, p, yerr=u, fmt='.', alpha=0.2, markersize=2, elinewidth=0.3, color='steelblue')
            lims = [min(t.min(), p.min()), max(t.max(), p.max())]
            mg = (lims[1]-lims[0])*0.05
            ax.plot([lims[0]-mg, lims[1]+mg], [lims[0]-mg, lims[1]+mg], 'k--', lw=1)
            ax.set_xlim(lims[0]-mg, lims[1]+mg); ax.set_ylim(lims[0]-mg, lims[1]+mg)
            sym = 'Om' if col==0 else 's8'
            ax.set_title(f'{mname} | R2={met[f"{param}_r2"]:.4f} RMSE={met[f"{param}_rmse"]:.4f}')
            ax.set_xlabel(f'True {sym}'); ax.set_ylabel(f'Pred {sym}')
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()

def plot_training_curves(histories, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for name, h in histories.items():
        axes[0].plot(h['val_loss'], label=name)
        axes[1].plot(h['val_r2_om'], label=name)
        axes[2].plot(h['val_r2_s8'], label=name)
    axes[0].set_title('Val Loss'); axes[1].set_title('R2 (Omega_m)'); axes[2].set_title('R2 (sigma_8)')
    for ax in axes: ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlabel('Epoch')
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── Keep-alive: prevents Kaggle inactivity timeout ──
# Method 1: JavaScript (injects browser-level activity that Kaggle detects)
from IPython.display import display, Javascript
display(Javascript('''
function kaggleKeepAlive() {
    // Simulate keyboard event — Kaggle's frontend treats this as user activity
    document.dispatchEvent(new KeyboardEvent('keypress', {key: 'a'}));
}
setInterval(kaggleKeepAlive, 30000);
console.log("keep-alive: pinging every 30s");
'''))
print('JS keep-alive injected (pings every 30s)')

---
## 6. EXPERIMENT 1: Core Benchmark

CNN vs ViT vs CosmoMamba on dark matter density, IllustrisTNG.

In [ ]:
train_loader, val_loader, test_loader = make_loaders(str(DATA_DIR), ['Mcdm'], 'IllustrisTNG', batch_size=32)

In [ ]:
cnn_model, cnn_hist = train_model(CNNBaseline(in_channels=1), train_loader, val_loader, name='cnn_Mcdm_TNG', epochs=300)

In [ ]:
vit_model, vit_hist = train_model(ViTRegressor(in_channels=1), train_loader, val_loader, name='vit_Mcdm_TNG', epochs=300)

In [ ]:
import gc
del cnn_model, vit_model
gc.collect()
torch.cuda.empty_cache()

mamba_train, mamba_val, _ = make_loaders(str(DATA_DIR), ['Mcdm'], 'IllustrisTNG', batch_size=16)
mamba_model, mamba_hist = train_model(
    CosmoMamba(in_channels=1, embed_dim=128, depth=6, patch_size=32),
    mamba_train, mamba_val, name='mamba_Mcdm_TNG', epochs=300)

In [ ]:
# Reload all three from disk (works after session expiry / Run All from scratch)
MAMBA_KW = dict(in_channels=1, embed_dim=128, depth=6, patch_size=32)

cnn_model = CNNBaseline(in_channels=1).to(DEVICE)
cnn_ckpt = torch.load(RUNS_DIR / 'cnn_Mcdm_TNG' / 'best_model.pt', map_location=DEVICE, weights_only=False)
cnn_model.load_state_dict(cnn_ckpt['model'])
cnn_hist = cnn_ckpt['history']

vit_model = ViTRegressor(in_channels=1).to(DEVICE)
vit_ckpt = torch.load(RUNS_DIR / 'vit_Mcdm_TNG' / 'best_model.pt', map_location=DEVICE, weights_only=False)
vit_model.load_state_dict(vit_ckpt['model'])
vit_hist = vit_ckpt['history']

mamba_model = CosmoMamba(**MAMBA_KW).to(DEVICE)
_mdir = RUNS_DIR / 'mamba_Mcdm_TNG'
_best, _resume = _mdir / 'best_model.pt', _mdir / 'resume.pt'
if _best.exists():
    _mc = torch.load(_best, map_location=DEVICE, weights_only=False)
    mamba_model.load_state_dict(_mc['model'])
    mamba_hist = _mc['history']
elif _resume.exists():
    _mr = torch.load(_resume, map_location=DEVICE, weights_only=False)
    mamba_model.load_state_dict(_mr['best_state'])
    mamba_hist = _mr.get('history', {'train_loss': [], 'val_loss': [], 'val_r2_om': [], 'val_r2_s8': []})
    print(f'Mamba: loaded best-so-far from resume.pt (last epoch {_mr["epoch"]}) — re-run training cell to continue')
else:
    raise FileNotFoundError(f'No Mamba checkpoint in {_mdir}')

print('\nTEST SET RESULTS')
print('='*80)
test_results = {}
for name, model in [('CNN', cnn_model), ('ViT', vit_model), ('CosmoMamba', mamba_model)]:
    preds, logvars, targets, metrics = test_model(model, test_loader, label=name)
    test_results[name] = (preds, logvars, targets, metrics)

plot_comparison(test_results, str(RUNS_DIR / 'benchmark_predictions.png'))
plot_training_curves({'CNN': cnn_hist, 'ViT': vit_hist, 'CosmoMamba': mamba_hist}, str(RUNS_DIR / 'training_curves.png'))

print(f'\n{"Model":12s} | {"Om R2":>8s} | {"s8 R2":>8s} | {"Om RMSE":>8s} | {"s8 RMSE":>8s}')
print('-'*55)
for nm, (_, _, _, m) in test_results.items():
    print(f'{nm:12s} | {m["Omega_m_r2"]:8.4f} | {m["sigma_8_r2"]:8.4f} | {m["Omega_m_rmse"]:8.5f} | {m["sigma_8_rmse"]:8.5f}')

---
## 7. EXPERIMENT 2: Cross-Suite Robustness

Trained on IllustrisTNG, tested on SIMBA **without retraining**.

In [ ]:
_, _, simba_test_loader = make_loaders(str(DATA_DIR), ['Mcdm'], 'SIMBA', batch_size=32)

print('CROSS-SUITE: IllustrisTNG -> SIMBA')
print('='*80)
cross_results = {}
for name, model in [('CNN', cnn_model), ('ViT', vit_model), ('CosmoMamba', mamba_model)]:
    preds, logvars, targets, metrics = test_model(model, simba_test_loader, label=f'{name} -> SIMBA')
    cross_results[f'{name} -> SIMBA'] = (preds, logvars, targets, metrics)

plot_comparison(cross_results, str(RUNS_DIR / 'cross_suite_predictions.png'))

print(f'\n{"Model":20s} | {"In-domain R2":>12s} | {"Cross-suite R2":>14s} | {"Delta":>8s}')
print('-'*65)
for name in ['CNN', 'ViT', 'CosmoMamba']:
    in_r2 = test_results[name][3]['mean_r2']
    cr_r2 = cross_results[f'{name} -> SIMBA'][3]['mean_r2']
    print(f'{name:20s} | {in_r2:12.4f} | {cr_r2:14.4f} | {cr_r2-in_r2:+8.4f}')

---
## 8. EXPERIMENT 3: Ablation Study

Effect of depth, d_state, and embed_dim on CosmoMamba.

In [ ]:
# Nuke everything from prior experiments to reclaim VRAM
import gc
_keep = set(dir()) | {'_keep'}
for _v in ['cnn_model','vit_model','mamba_model','cnn_ckpt','vit_ckpt',
           'test_results','cross_results','train_loader','val_loader',
           'test_loader','simba_test_loader','mamba_train','mamba_val']:
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

ablation_configs = {
    'depth_3':   dict(embed_dim=128, depth=3,  d_state=16, patch_size=32),
    'depth_6':   dict(embed_dim=128, depth=6,  d_state=16, patch_size=32),
    'dstate_8':  dict(embed_dim=128, depth=6,  d_state=8,  patch_size=32),
    'dim_96':    dict(embed_dim=96,  depth=6,  d_state=16, patch_size=32),
}

abl_train, abl_val, abl_test = make_loaders(str(DATA_DIR), ['Mcdm'], 'IllustrisTNG', batch_size=4)
ablation_results = {}
for abl_name, kwargs in ablation_configs.items():
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n--- {abl_name} (GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB) ---')
    ckpt_path = RUNS_DIR / f'ablation_{abl_name}' / 'best_model.pt'
    if ckpt_path.exists():
        print(f'  Skipping {abl_name} (checkpoint exists)')
        m = CosmoMamba(in_channels=1, **kwargs).to(DEVICE)
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False)['model'])
    else:
        m, _ = train_model(CosmoMamba(in_channels=1, **kwargs), abl_train, abl_val,
                           name=f'ablation_{abl_name}', epochs=150, patience=25)
    _, _, _, metrics = test_model(m, abl_test, label=f'ablation/{abl_name}')
    n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    ablation_results[abl_name] = {**metrics, 'params_M': n_p / 1e6}
    del m
    gc.collect(); torch.cuda.empty_cache()

print(f'\n{"Config":12s} | {"Params":>7s} | {"Om R2":>8s} | {"s8 R2":>8s} | {"Om RMSE":>9s} | {"s8 RMSE":>9s}')
print('-'*70)
for nm, m in ablation_results.items():
    print(f'{nm:12s} | {m["params_M"]:6.2f}M | {m["Omega_m_r2"]:8.4f} | {m["sigma_8_r2"]:8.4f} | {m["Omega_m_rmse"]:9.5f} | {m["sigma_8_rmse"]:9.5f}')

## 9. Save Results

In [ ]:
all_results = {
    'benchmark':  {n: {k: float(v) for k,v in m.items()} for n,(_,_,_,m) in test_results.items()},
    'cross_suite':{n: {k: float(v) for k,v in m.items()} for n,(_,_,_,m) in cross_results.items()},
}
if ablation_results:
    all_results['ablations'] = {k: {kk: float(vv) for kk,vv in v.items()} for k,v in ablation_results.items()}

with open(RUNS_DIR / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('Saved! Files:')
for p in sorted(RUNS_DIR.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(RUNS_DIR)} ({p.stat().st_size/1e6:.1f} MB)')